# 04 — Exact circuit synthesis with `TreeMPU`

`TreeMPU` implements the tree decomposition of Styliaris, Trivedi and Cirac (PRL **135**, 260602) with the extensions of the thesis:

1. cut the chain into site isometries $V_n=L\,A\,R$ using boundary operators $L,R$;
2. merge neighbours pairwise along a binary tree; each merge block-encodes the rank-one merging operator with LCU norm
   $$q=\|R^{-\top}L^{-1}\|_F;$$
3. make each merge exact with $\ell$ rounds of oblivious amplitude amplification, where $\ell$ is the smallest integer with $q\le\csc\frac{\pi}{4\ell+2}$ (so $q\le2\Rightarrow\ell=1$, $q\le3.236\Rightarrow\ell=2$).

The circuit depth then scales as $N^{\log_2(2\ell+1)}$, which is why small $q$ matters.

In [1]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")  # TreeMPU's optimizers run on the CPU

import numpy as np
from qiskit import transpile

from TreeMPU import (
    UniformMPU,
    dense_unitary_from_tensors,
    phase_aligned_error,
    reconstruct_circuit_unitary,
)

A = np.zeros((2, 2, 2, 2), dtype=complex)  # MCZ tensor from notebook 01
A[0, 0] = np.diag([1, 0])
A[1, 1] = np.eye(2)
l_vec, r_vec = np.array([1, 1], dtype=complex), np.array([1, -2], dtype=complex)

mpu = UniformMPU(A=A, l_vec=l_vec, r_vec=r_vec, N=8, light=True)
print(f"q = {mpu.q_unif:.6f}, ell = {mpu.ell}, boundary = {mpu.boundary_source}")

q = 2.000000, ell = 1, boundary = canonical


## Synthesizing the circuit

`synthesize()` returns the circuit and its registers. The circuit contains *composite* gates (multi-qubit unitaries and isometries); transpile it for elementary-gate counts.

In [2]:
qc, sys_regs, bond_regs, anc_regs, ancillas = mpu.synthesize()
print(f"composite circuit: {qc.num_qubits} qubits, depth {qc.depth()}")
qc_elem = transpile(qc, basis_gates=["u", "cx"], optimization_level=1)
print(
    f"elementary [u, cx]: depth {qc_elem.depth()}, gates {sum(qc_elem.count_ops().values())}, "
    f"CX {qc_elem.count_ops().get('cx', 0)}"
)

composite circuit: 36 qubits, depth 134
elementary [u, cx]: depth 2018, gates 10086, CX 4048


## Verifying the circuit

`reconstruct_circuit_unitary` simulates the circuit on every computational-basis input of the system registers, with all other qubits in $|0\rangle$, and post-selects them on $|0\rangle$. For an exact synthesis it returns $U_N$ up to a global phase, with zero leakage. This is feasible for small circuits (up to about 24 qubits).

In [3]:
small = UniformMPU(A=A, l_vec=l_vec, r_vec=r_vec, N=4)
qc4, sys4, *_ = small.synthesize()
U_rec, leakage = reconstruct_circuit_unitary(qc4, sys4)
U_ref = dense_unitary_from_tensors([A] * 4, l_vec, r_vec)
print(
    f"{qc4.num_qubits} qubits | error (up to global phase) = {phase_aligned_error(U_rec, U_ref):.1e} | leakage = {leakage:.1e}"
)

16 qubits | error (up to global phase) = 1.3e-14 | leakage = 2.7e-14


## Boundary strategies and the conditioning number

The boundary pair decides $q$, and hence $\ell$ and the depth. `UniformMPU` tries, in order:

| option | `boundary_source` | result |
|---|---|---|
| `boundary_mode1_exact=True` | `mode1_exact` | closed form, $q=\chi$ (Mode I, single-vertex readout) |
| `boundary_opt_k=k` | `qD_general` / `optimized[_gauge]` | $q=\chi$ solver, else direct optimization |
| default | `canonical_sigma_tau` / `canonical` | canonical boundaries (optimized $\sigma,\tau$ kept only if they lower $q$) |

For a random Mode I tensor the canonical boundary gives $q>2$, hence $\ell=2$; the $q=\chi$ constructions reach $q=2$, hence $\ell=1$:

In [4]:
from mpu_generators import build_family

t, l, r = build_family("branching_uniform", d=2, bond_dim=2, seed=0)
for options in [
    dict(boundary_sigma_tau=False),
    dict(),
    dict(boundary_mode1_exact=True),
    dict(boundary_opt_k=1),
]:
    m = UniformMPU(A=t[0], l_vec=l, r_vec=r, N=4, light=True, **options)
    print(f"{str(options):32s} -> {m.boundary_source:20s} q = {m.q_unif:.6f}  ell = {m.ell}")

{'boundary_sigma_tau': False}    -> canonical            q = 2.064124  ell = 2


{}                               -> canonical            q = 2.064124  ell = 2


{'boundary_mode1_exact': True}   -> mode1_exact          q = 2.000000  ell = 1


{'boundary_opt_k': 1}            -> qD_general           q = 2.000000  ell = 1


Boundaries from iterative solvers are accepted at isometry tolerance $10^{-4}$. `block_isometry_residuals` measures $\|V^\dagger V-\mathbb 1\|$ for boundary and bulk blocks of several lengths. Here the `qD_general` boundary is isometric to machine precision at every length:

In [5]:
from TreeMPU import block_isometry_residuals

m = UniformMPU(A=t[0], l_vec=l, r_vec=r, N=4, light=True, boundary_opt_k=1)
for n, (left, bulk, right) in block_isometry_residuals(
    t[0], l, r, m.L, m.R, lengths=(1, 2, 4, 8)
).items():
    print(f"block length {n}: left {left:.1e}  bulk {bulk:.1e}  right {right:.1e}")

block length 1: left 1.9e-15  bulk 3.6e-15  right 1.2e-15
block length 2: left 2.8e-15  bulk 5.0e-15  right 2.5e-15
block length 4: left 6.3e-15  bulk 8.7e-15  right 5.8e-15
block length 8: left 4.1e-14  bulk 3.4e-14  right 3.4e-14


**Next:** notebook 05 estimates resources at large $N$ without building circuits, and treats site-dependent MPUs.